# Module 4.3: The Full Transformer

Welcome to the Grand Finale of Architecture Assembly! We have successfully built the math, the tokens, the positional wave functions, the powerful Encoder (Reader), and the autoregressive Decoder (Writer).

In this notebook, we wire the electrical box together and launch the complete 2017 `Seq2Seq` (Sequence-to-Sequence) Transformer.

> ⚠️ **Reminder (see Module 4.1):** this is the *historical* 2017 Encoder–Decoder Transformer, built so you can see every part. From Module 4.3 onward we keep **only the decoder stack** (Llama-style) and delete the Encoder + Cross-Attention. This notebook is the "complete picture" before we simplify.

## 1. The Grand Assembly (Stacking Layers)

### The Concept
A real Transformer is not just one Encoder block and one Decoder block. It is a stack of them. The original 2017 paper used **6** stacked layers in each half. Frontier models are much deeper — the exact layer counts for closed models like GPT-4 are not public, but they are reported to use *roughly* dozens-to-a-hundred layers.

> **Heads up on numbers in this notebook:** the diagram below draws **3** layers (to keep it readable), the stacking demo creates **6**, and the final model uses **4**. The layer count is just a configurable hyperparameter (`num_layers`) — none of these is "the" right number.

Below is the architectural flowchart of data moving through our network:

```mermaid
graph TD
    A[French Text: Bonjour] --> B[Token/Embedding + Position]
    B --> C[Encoder Block 1]
    C --> D[Encoder Block 2]
    D --> E[Encoder Block 3]
    
    F[English Prefix: Hello] --> G[Token/Embedding + Position]
    G --> H[Decoder Block 1]
    H --> I[Decoder Block 2]
    I --> J[Decoder Block 3]

    E -.->|Cross-Attention| H
    E -.->|Cross-Attention| I
    E -.->|Cross-Attention| J
```

### Why do we need it?
One layer isn't deep enough to reason. Researchers who probe trained networks have observed a *rough* progression: early layers tend to capture surface features (like part-of-speech), while later layers capture more abstract, task-relevant information. The reality is messier than a clean "layer N = concept N" story — features are distributed and overlapping — but stacking depth clearly lets the model build increasingly abstract representations.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)  # Reproducibility

# In PyTorch, we use nn.ModuleList() to create a stack of layers.
# IMPORTANT: create a NEW module instance for each layer.
# If you reuse the SAME object N times, all "layers" share ONE set of weights,
# so the stack can't learn N distinct transformations (a classic bug!).
stacked_layers = nn.ModuleList([nn.Linear(128, 128) for _ in range(6)])

print(f"We have stacked {len(stacked_layers)} layers!")

# Proof they are independent: each layer is a distinct object with its own weights.
ids = {id(layer) for layer in stacked_layers}
print(f"Distinct layer objects: {len(ids)} (should be 6, not 1)")

## 2. The Final Output (Linear + Softmax)

### The Concept
When the final Decoder Block finishes, it outputs a dense vector (e.g. `128` floating-point numbers). But that is useless to humans. We need it to pick an actual English word from our Vocabulary Dictionary (which might have `50,000` words)!

```mermaid
graph LR
    A[Decoder Output Vector\nDim: 128] -->|Linear Matrix| B[Logits\nDim: 50,000]
    B -->|Softmax| C[Probabilities\nDim: 50,000]
    C -->|argmax / sample| D[Word: 'friend']
```

### Why do we need it? (Closing the Loop)
We use a massive `nn.Linear` matrix to stretch the `128` features into `50,000` raw scores (**logits**). Passing logits through **softmax** turns them into a probability distribution over the whole vocabulary, and then we pick a token (here, with `argmax`; real models often *sample*). This closes the loop back to the **Softmax Math** from Module 1.1!

In [ ]:
import torch.nn.functional as F

torch.manual_seed(0)
vocab_size = 50000
decoder_vector = torch.randn(128)          # one final decoder output vector
to_vocab = nn.Linear(128, vocab_size)      # the "unembedding" projection

logits = to_vocab(decoder_vector)          # raw scores, can be any real number
probs = F.softmax(logits, dim=-1)          # turn logits into probabilities that sum to 1

predicted_token_id = int(torch.argmax(probs))
print(f"Logits shape: {logits.shape} (one raw score per vocabulary word)")
print(f"Probabilities sum to: {probs.sum().item():.4f} (always 1.0 after softmax)")
print(f"Highest-probability token id: {predicted_token_id}")
print(f"Its probability: {probs[predicted_token_id].item():.6f}")
print("\n(In a real model, token id {} would map back to an actual word via the tokenizer.)".format(predicted_token_id))

## 3. Building the End-To-End Transformer!

Let's write out the ultimate class.

*(Note: For this notebook to run cleanly and instantly, we **mock** the interior Encoder/Decoder blocks — they just pass tensors through. But the surrounding plumbing matches the real code, and the mocked `DecoderBlock` uses the **exact same `forward` signature** — `(x, encoder_output, causal_mask)` — as the real `DecoderBlock` from Module 4.2, so the blocks are genuinely drop-in compatible.)*

In [ ]:
# --- Mocks standing in for the real blocks from notebooks 08 and 09 ---
class MockEncoderBlock(nn.Module):
    def forward(self, x):
        # Real version runs Self-Attention + FFN (notebook 08).
        return x

class MockDecoderBlock(nn.Module):
    # Signature matches the REAL DecoderBlock.forward from notebook 09 exactly:
    #   forward(self, x, encoder_output, causal_mask)
    def forward(self, x, encoder_output, causal_mask):
        # Real version runs Masked Self-Attention + Cross-Attention + FFN (notebook 09).
        return x
# ---------------------------------------------------------------------

def positional_encoding(x):
    # Placeholder: in the real pipeline we ADD positional encodings (notebook 06)
    # to the token embeddings so the model knows word order. Returning x unchanged
    # here keeps the demo simple, but the line is left in so the assembly is honest.
    return x  # TODO: replace with real sinusoidal/RoPE positional encoding

class Transformer(nn.Module):
    def __init__(self,
                 src_vocab_size: int,
                 tgt_vocab_size: int,
                 d_model: int = 256,
                 num_layers: int = 6):
        super().__init__()

        # 1. Embeddings & Positions
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)

        # 2. The Stacks -- a NEW instance per layer (no shared-weights bug!)
        self.encoder_stack = nn.ModuleList([MockEncoderBlock() for _ in range(num_layers)])
        self.decoder_stack = nn.ModuleList([MockDecoderBlock() for _ in range(num_layers)])

        # 3. The Final Vocabulary Projection
        self.final_linear = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt, tgt_mask):
        # Step A: Encoder reads the source text
        enc_x = self.src_embedding(src)
        enc_x = enc_x + positional_encoding(enc_x)  # add positional info (placeholder)
        for encoder in self.encoder_stack:
            enc_x = encoder(enc_x)

        # Step B: Decoder generates the target text (masked + cross-attending to the encoder)
        dec_x = self.tgt_embedding(tgt)
        dec_x = dec_x + positional_encoding(dec_x)  # add positional info (placeholder)
        for decoder in self.decoder_stack:
            # NOTE: keyword names match the real DecoderBlock.forward signature exactly.
            dec_x = decoder(dec_x, encoder_output=enc_x, causal_mask=tgt_mask)

        # Step C: Final Projection to Logits
        logits = self.final_linear(dec_x)

        return logits

# --- TEST TIME ---
# Pretend we have a dataset with 10k French words and 15k English words.
torch.manual_seed(0)
model = Transformer(src_vocab_size=10000, tgt_vocab_size=15000, d_model=256, num_layers=4)

# Send in 1 sentence, 10 French words
french_tokens = torch.randint(0, 10000, (1, 10))
# Send in the English generation prefix (3 words so far)
english_prefix = torch.randint(0, 15000, (1, 3))
mask = torch.tril(torch.ones(3, 3))

final_prediction_logits = model(french_tokens, english_prefix, mask)
print(f"Input shape (English Prompt): {english_prefix.shape}")
print(f"Output Logits shape: {final_prediction_logits.shape} -> (Batch, Target_Seq, Vocab_Size!)")

# And to turn the LAST position's logits into an actual predicted token:
last_step_logits = final_prediction_logits[0, -1]      # logits for the next word
next_probs = F.softmax(last_step_logits, dim=-1)
print(f"Predicted next-token id: {int(torch.argmax(next_probs))}")

## 4. The Modern Split: Encoder-Only vs Decoder-Only

The architecture we just built is massive. It handles Sequence-to-Sequence (like translation or summarizing). But modern AI has realized we can just physically split the architecture in half to specialize in different tasks!

```mermaid
graph TD
    A[Sequence-to-Sequence\nThe Original 2017 Transformer] 
    A -->|Delete Decoder| B[Encoder-Only Architecture]
    A -->|Delete Encoder| C[Decoder-Only Architecture]
    
    B --> D{Google BERT}
    C --> E{ChatGPT / Llama 3}
    
    D -.->|Use Cases| F(Sentence Classification\nUnderstanding text\nSearching databases)
    E -.->|Use Cases| G(Talking to Humans\nGenerating Python Code\nWriting Essays)
```

### Why do we need it?
1. **Encoder-only (BERT)**: If you just need to classify a spam email, you don't need to generate text. You just need a model that reads the *entire* email at once and outputs a `True` or `False`. Encoder-only is perfectly optimized for this.
2. **Decoder-Only (GPT)**: If you want to chat with a bot, it doesn't need to translate anything. It just needs to aggressively read history and predict the very next word, over and over again. Decoder-Only models are leaner, faster, and scale incredibly well on modern GPUs.

> **This is the fork in the road.** From Module 4.3 onward we follow path **C** (Decoder-Only): we throw away the Encoder and Cross-Attention, and build up the Llama-style stack for real.

### 🏋️ Try it yourself

**Task 1 — Make depth configurable.** Build two `Transformer`s, one with `num_layers=2` and one with `num_layers=12`. Print the total parameter count of each (`sum(p.numel() for p in model.parameters())`). How much does depth cost?

**Task 2 — Greedy decode one step.** Using the trained-looking `model` above, take `final_prediction_logits`, apply softmax to the last position, and print the top-5 most likely next-token ids with their probabilities (hint: `torch.topk`).

In [ ]:
# Task 1 starter:
# for n in (2, 12):
#     m = Transformer(10000, 15000, d_model=256, num_layers=n)
#     print(f"num_layers={n}: {sum(p.numel() for p in m.parameters()):,} params")

# Task 2 starter:
last_logits = final_prediction_logits[0, -1]
probs = F.softmax(last_logits, dim=-1)
top_probs, top_ids = torch.topk(probs, k=5)
for rank, (p, tid) in enumerate(zip(top_probs.tolist(), top_ids.tolist()), start=1):
    print(f"#{rank}: token id {tid:5d}  prob {p:.6f}")